# Week 08 - Tree Ensemble and Neural Network

**Project:** AI/ML-Augmented Black-Scholes Framework
**Mandatory output of master plan section 11.8.**
**Stage Gate 2**, 17 November 2026.

Milestone:

> The student shall identify whether additional nonlinear model complexity
> produces a **meaningful** improvement over regression.

The word doing the work is *meaningful*. On noiseless synthetic data any
flexible model will beat the regression baselines on RMSE, so a ranking
settles nothing. This notebook therefore reports three things for every
model: accuracy, the residual structure on identical buckets, and the
financial consistency checks of section 5.28 as **counts of violations**.

Everything imports from modules that carry their own self-tests, so the
notebook and `postmidsem_week7.py` cannot drift apart.

In [ ]:
import sys, os
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.getcwd())

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams.update({"figure.figsize": (11, 3.8), "axes.grid": True,
                     "grid.alpha": 0.3, "font.size": 10,
                     "axes.spines.top": False, "axes.spines.right": False})
print("working directory:", os.getcwd())

from ml_dataset import SEED, FEATURES_BASIC, load_dataset, get_xy
from linear_models import Standardiser, fit_ols, select_ridge_alpha, metrics
from tree_models import RandomForest
from neural_net import MLP
from black_scholes import bs_call
import financial_checks as fc

K_REF, Q_REF = 100.0, 0.015
df = load_dataset()
print("frozen Week 6 split:", df["split"].value_counts().to_dict())

## 1. The models

Same features, same target, same split, same scaler as Week 6. The scaler is
fitted on training rows only; Ridge `alpha` and the network's stopping epoch
are chosen on validation; the test split is scored once at the end.

Both the forest and the network are written out in this repository rather than
imported from a library, because master plan 5.31 requires an answer to "what
is being minimised, with what initialisation and what hyperparameters" for
every model. Both are cross-checked where a library equivalent exists.

In [ ]:
Xtr, ytr = get_xy(df, FEATURES_BASIC, split="train")
Xva, yva = get_xy(df, FEATURES_BASIC, split="val")
Xte, yte = get_xy(df, FEATURES_BASIC, split="test")

sc = Standardiser().fit(Xtr)                 # TRAIN ONLY
Ztr, Zva, Zte = sc.transform(Xtr), sc.transform(Xva), sc.transform(Xte)

ols = fit_ols(Ztr, ytr, feature_names=FEATURES_BASIC)
alpha, ridge, _ = select_ridge_alpha(Ztr, ytr, Zva, yva)
print("ridge alpha (validation):", alpha)

In [ ]:
def expand(X):
    cols, p = [X], X.shape[1]
    for i in range(p):
        for j in range(i, p):
            cols.append((X[:, i] * X[:, j])[:, None])
    return np.hstack(cols)

scq = Standardiser().fit(expand(Xtr))
olsq = fit_ols(scq.transform(expand(Xtr)), ytr)
print("quadratic baseline fitted:", expand(Xtr).shape[1], "terms")

In [ ]:
%%time
# Trees are invariant to monotone transforms of a feature, so the forest is
# fitted on RAW features - standardising would change nothing.
rf = RandomForest(n_trees=60, max_depth=14, min_samples_leaf=5,
                  max_features=3, seed=SEED).fit(Xtr, ytr)
print("OOB RMSE %.4f (currency)"
      % (K_REF * np.sqrt(np.nanmean((rf.oob_prediction_ - ytr) ** 2))))

In [ ]:
%%time
net = MLP(4, hidden=(32, 16), lr=3e-3, batch=256, epochs=400,
          seed=SEED, patience=50).fit(Ztr, ytr, Zva, yva)
print("best epoch", net.best_epoch_)

# The constrained variant. ML_experiment_plan.md 9.1 PREDICTED, before any
# fitting, that an unconstrained linear output would price the deep wings
# below zero, and committed to fixing it in the specification rather than by
# clipping. softplus makes C >= 0 impossible to violate.
netc = MLP(4, hidden=(32, 16), lr=3e-3, batch=256, epochs=400,
           seed=SEED, patience=50, output="softplus").fit(Ztr, ytr, Zva, yva)
print("best epoch (softplus)", netc.best_epoch_)

In [ ]:
def bs_bench(X):
    m, T, r, sg = X[:, 0], X[:, 1], X[:, 2], X[:, 3]
    return bs_call(K_REF * np.exp(m), K_REF, T, r, sg, Q_REF) / K_REF

models = {
    "OLS":                   lambda X: ols.predict(sc.transform(X)),
    "Ridge":                 lambda X: ridge.predict(sc.transform(X)),
    "OLS quadratic":         lambda X: olsq.predict(scq.transform(expand(X))),
    "Random forest":         lambda X: rf.predict(X),
    "Neural net":            lambda X: net.predict(sc.transform(X)),
    "Neural net (softplus)": lambda X: netc.predict(sc.transform(X)),
    "Black-Scholes":         bs_bench,
}
print(len(models), "models")

## 2. Required comparison: MAE, RMSE, R-squared

The test split is scored here for the first and only time.

In [ ]:
rows = []
for name, f in models.items():
    m_ = metrics(yte, f(Xte), scale=K_REF)
    m_["model"] = name
    rows.append(m_)
tab = pd.DataFrame(rows)[["model", "MAE", "RMSE", "R2", "max_abs_error"]]
tab.round(4)

Black-Scholes has an RMSE of order `1e-14` - machine precision, because
it *is* the data-generating process. No model can beat it, and one that
appeared to would be a bug. The ranking below it measures capacity, not
usefulness.

## 3. Mandatory plot 1: predicted versus true

In [ ]:
order = ["OLS", "OLS quadratic", "Random forest", "Neural net"]
fig, ax = plt.subplots(1, 4, figsize=(15, 3.6), sharex=True, sharey=True)
lim = [0, yte.max() * K_REF * 1.02]
for a, nm in zip(ax, order):
    a.scatter(yte * K_REF, models[nm](Xte) * K_REF, s=3, alpha=0.25,
              color="#4878a8")
    a.plot(lim, lim, color="black", lw=1)
    a.set_title(nm); a.set_xlabel("true price")
ax[0].set_ylabel("predicted price")
plt.tight_layout(); plt.show()

## 4. Mandatory plots 2 and 3: residual versus moneyness and maturity

Master plan 5.29: *"The student shall not report only one global error
statistic."* The same buckets as Week 6, so models are comparable bucket by
bucket.

In [ ]:
sub = df[df.split == "test"]
fig, ax = plt.subplots(2, 4, figsize=(15, 6.2), sharex="row", sharey="row")
for j, nm in enumerate(order):
    e = (models[nm](Xte) - yte) * K_REF
    ax[0, j].axhline(0, color="black", lw=1)
    ax[0, j].scatter(sub["m"], e, s=3, alpha=0.25, color="#b5483d")
    ax[0, j].set_title(nm); ax[0, j].set_xlabel("m")
    ax[1, j].axhline(0, color="black", lw=1)
    ax[1, j].scatter(sub["T"], e, s=3, alpha=0.25, color="#b5483d")
    ax[1, j].set_xscale("log"); ax[1, j].set_xlabel("T (log)")
ax[0, 0].set_ylabel("residual vs m"); ax[1, 0].set_ylabel("residual vs T")
plt.tight_layout(); plt.show()

In [ ]:
frames = []
for nm, f in models.items():
    if nm == "Black-Scholes":
        continue
    d = pd.DataFrame({"m": sub["m"].to_numpy(),
                      "e": (f(Xte) - yte) * K_REF})
    d["bin"] = pd.cut(d["m"], [-0.61, -0.4, -0.2, -0.05, 0.05, 0.2, 0.4, 0.61])
    g = d.groupby("bin", observed=True)["e"].agg(
        mean_abs=lambda s: s.abs().mean())
    g.columns = [nm]
    frames.append(g)
pd.concat(frames, axis=1).round(4)

## 5. Mandatory plot 4: feature importance

In [ ]:
imp = rf.importance_
idx = np.argsort(imp)[::-1]
fig, ax = plt.subplots(figsize=(6, 3.4))
ax.bar([FEATURES_BASIC[i] for i in idx], imp[idx], color="#4878a8")
for i, v in enumerate(imp[idx]):
    ax.text(i, v + 0.01, "%.3f" % v, ha="center", fontsize=9)
ax.set_ylabel("share of variance reduction")
ax.set_title("random forest feature importance")
plt.tight_layout(); plt.show()

## 6. Mandatory plot 5: training and validation loss

In [ ]:
h = net.history_
fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
ax[0].plot(h["epoch"], h["train"], label="train")
ax[0].plot(h["epoch"], h["val"], label="validation")
ax[0].axvline(net.best_epoch_, ls=":", color="black",
              label="best epoch %d" % net.best_epoch_)
ax[0].set_yscale("log"); ax[0].set_xlabel("epoch")
ax[0].set_ylabel("MSE (scaled target)"); ax[0].legend(fontsize=8)
ax[0].set_title("unconstrained network")
hc = netc.history_
ax[1].plot(hc["epoch"], hc["train"], label="train")
ax[1].plot(hc["epoch"], hc["val"], label="validation")
ax[1].set_yscale("log"); ax[1].set_xlabel("epoch"); ax[1].legend(fontsize=8)
ax[1].set_title("softplus output")
plt.tight_layout(); plt.show()

The validation curve sits on top of the training curve throughout.
With 14,000 rows, four features and roughly 700 parameters there is no
overfitting to speak of - and on a noiseless target there is no noise to
overfit *to*. Early stopping is therefore doing very little here. It is kept
because it will matter on market data, where the target is noisy.

## 7. Financial consistency - the part that decides the gate

Master plan 5.28, reported as **counts**. An average hides a model that
prices one point in fifty below zero.

Checks 1-3 are pointwise on the test rows. Checks 4-6 are statements about
the *fitted function*, so they are swept on a grid holding other inputs
fixed - which also probes regions the test split does not cover.

In [ ]:
S = sub["S"].to_numpy(); K = sub["K"].to_numpy()
T = sub["T"].to_numpy(); r = sub["r"].to_numpy()
q = sub["q"].to_numpy(); m = sub["m"].to_numpy()

tabs = []
for nm, f in models.items():
    pw = fc.pointwise_checks(f(Xte), S, K, T, r, q, m=m)
    sw = fc.sweep_checks(f, n_grid=161)
    tabs.append(fc.check_table(nm, pw, sw))
fin = pd.concat(tabs, ignore_index=True)
fin.pivot(index="model", columns="check", values="n_violations")

### Reading the table

**Black-Scholes is clean on all six**, which is what validates the framework:
if the true pricer were flagged, the checker would be wrong.

**OLS and Ridge fail the butterfly condition at 100% of grid points**, and
that is a theorem rather than an accident. With `y = C/K` linear in
`m = ln(S/K)`,

$$C(K) = K\,y(m), \qquad \frac{d^2C}{dK^2} = \frac{y'' - y'}{K} = -\frac{dy/dm}{K} < 0$$

since `y'' = 0` and `dy/dm > 0`. A model linear in log-moneyness is
**necessarily concave in strike**, so a butterfly spread priced off it always
has negative value. No amount of RMSE would reveal that.

**The random forest never predicts a negative price** - it cannot, because
every prediction is an average of training targets and those are all
non-negative. The same property makes it fail convexity badly: a step function
is locally neither convex nor concave.

**The unconstrained network has the best RMSE and the worst behaviour**,
which is exactly the risk flagged in `ML_experiment_plan.md` section 9.1
before any model was fitted.

In [ ]:
nn  = float(fin[(fin.model == "Neural net") & (fin.check == "C >= 0")].n_violations.iloc[0])
nnc = float(fin[(fin.model == "Neural net (softplus)") & (fin.check == "C >= 0")].n_violations.iloc[0])
r_un = float(tab.loc[tab.model == "Neural net", "RMSE"].iloc[0])
r_co = float(tab.loc[tab.model == "Neural net (softplus)", "RMSE"].iloc[0])
print("unconstrained:  RMSE %.4f   negative prices %4d" % (r_un, nn))
print("softplus:       RMSE %.4f   negative prices %4d" % (r_co, nnc))
print()
print("The constraint removed every negative price AND improved accuracy by")
print("%.0f%%. It is not a trade-off: softplus encodes a true property of the" % (100*(1-r_co/r_un)))
print("target, so it removes a degree of freedom the model was wasting.")

## 8. The gate question

> **Does additional nonlinear model complexity produce a meaningful
> improvement over regression?**

**On accuracy, yes, and it was never in doubt** - the target is noiseless
Black-Scholes, so capacity alone guarantees it. The honest baseline is the
quadratic expansion of the same four features, not OLS, because polynomial
regression is nearly free.

**On financial consistency, the ranking reverses.** The best-RMSE model is the
worst-behaved one. Complexity bought accuracy and cost structural correctness,
and only a check reported as counts makes that visible.

**The useful answer is the third one.** Imposing a known property of the
target in the model specification - a non-negative output - gave both
correctness and better accuracy at once. That is the finding to carry into the
real-data stage, where the target is noisy and the temptation to chase RMSE
will be far stronger.